# 04 · 用 Value 搭一个神经网络并训练

> **本节属于 Part 2 · 标量自动求导。**

上一节我们造好了自动求导引擎 `Value`。这一节我们用它**搭出真正的神经网络**（`Neuron / Layer / MLP`），训练一个二分类模型，并亲眼看到 autograd 的威力——以及它作为**标量**引擎的硬伤。

## 学习目标

- 用 `Value` 组装 `Neuron`（神经元）、`Layer`（层）、`MLP`（多层感知机）
- 实现参数收集 `parameters()` 与标准训练循环（`zero_grad → backward → step`）
- 训练一个二分类器并可视化**决策边界**
- 亲身体会**标量引擎为什么慢**，从而理解 Part 3 升级到"张量"的必要性

## 直觉与数学原理

- **神经元 (Neuron)**：对输入做加权和再过激活，$o = \sigma\!\big(\sum_i w_i x_i + b\big)$。
- **层 (Layer)**：一排并列的神经元。
- **MLP**：若干层堆叠，前一层的输出是后一层的输入；最后一层通常不加激活（输出原始分数）。

我们让每个 `w_i, b` 都是一个 `Value`，于是整个网络的前向计算会自动构成一张大计算图，`loss.backward()` 一次就能求出**所有**参数的梯度——无需任何手推！

## 从零手写实现：Neuron / Layer / MLP

In [ ]:
import random
from minitorch.scalar import Value
from minitorch import set_seed

class Neuron:
    def __init__(self, nin, nonlin=True):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)  # Σ w_i x_i + b
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout, **kw):
        self.neurons = [Neuron(nin, **kw) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        # 最后一层不加激活（输出原始分数）
        self.layers = [Layer(sz[i], sz[i + 1], nonlin=(i != len(nouts) - 1))
                       for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

set_seed(1)
model = MLP(2, [16, 1])              # 2 维输入 -> 隐藏层16 -> 输出1
print("参数数量:", len(model.parameters()))

## 造一批"双月牙"数据

经典的非线性二分类数据集 `make_moons`——两个交错的半月形。我们用约 20 行 NumPy 自己合成（不依赖 scikit-learn），标签取 $\{-1, +1\}$。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def make_moons(n=100, noise=0.1, seed=0):
    rng = np.random.RandomState(seed)
    n_out, n_in = n // 2, n - n // 2
    t_out = np.linspace(0, np.pi, n_out)
    t_in = np.linspace(0, np.pi, n_in)
    outer = np.c_[np.cos(t_out), np.sin(t_out)]
    inner = np.c_[1 - np.cos(t_in), 1 - np.sin(t_in) - 0.5]
    X = np.vstack([outer, inner]) + rng.randn(n, 2) * noise
    y = np.array([-1.0] * n_out + [1.0] * n_in)   # 标签 {-1, +1}
    return X, y

X, y = make_moons(n=100, noise=0.1)

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", s=18, alpha=0.8)
plt.title("make_moons (two classes)"); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 损失函数

二分类我们用 **max-margin (SVM hinge) 损失**：希望正确类别的分数 $y_i\cdot s_i \ge 1$，否则按差距罚分。再加一点 L2 正则防止权重过大：

$$L = \frac{1}{n}\sum_i \max(0,\; 1 - y_i s_i) \;+\; \alpha \sum_p p^2$$

In [ ]:
def compute_loss(model, X, y, alpha=1e-4):
    inputs = [[Value(xi) for xi in row] for row in X]
    scores = [model(inp) for inp in inputs]                     # 每个样本的分数
    margins = [(1 + (-yi) * si).relu() for yi, si in zip(y, scores)]
    data_loss = sum(margins) * (1.0 / len(margins))
    reg_loss = alpha * sum((p * p for p in model.parameters()))
    total = data_loss + reg_loss
    acc = sum((yi > 0) == (si.data > 0) for yi, si in zip(y, scores)) / len(y)
    return total, acc

## 训练

标准循环：**算损失 → 梯度清零 → backward → 参数更新**。是不是和 nb01 的线性回归一模一样？只是现在梯度由 `Value` 自动求出。

In [ ]:
import time

set_seed(1)
model = MLP(2, [16, 1])

t0 = time.time()
steps = 50
for k in range(steps):
    total, acc = compute_loss(model, X, y)
    for p in model.parameters():        # zero_grad
        p.grad = 0.0
    total.backward()                    # 自动反向
    lr = 1.0 - 0.9 * k / steps          # 学习率线性衰减
    for p in model.parameters():        # step
        p.data -= lr * p.grad
    if k % 10 == 0 or k == steps - 1:
        print(f"step {k:2d}  loss {total.data:.4f}  acc {acc*100:.0f}%")

print(f"\n训练 {steps} 步用时 {time.time() - t0:.2f}s")

## 看决策边界

为了画边界要在网格上密集地预测。这里我们把学到的参数取成普通 NumPy 数组、用矩阵运算快速前向（避免为每个网格点都建一张 `Value` 计算图）。

In [ ]:
def forward_numpy(model, A):
    """用学到的参数，以纯 NumPy 做前向（仅用于快速可视化）。"""
    for layer in model.layers:
        W = np.array([[w.data for w in nrn.w] for nrn in layer.neurons])  # (nout, nin)
        b = np.array([nrn.b.data for nrn in layer.neurons])
        Z = A @ W.T + b
        A = np.tanh(Z) if layer.neurons[0].nonlin else Z
    return A.ravel()

# 网格
xx, yy = np.meshgrid(np.linspace(-1.5, 2.5, 200), np.linspace(-1.0, 1.5, 200))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = forward_numpy(model, grid).reshape(xx.shape)

plt.figure(figsize=(5.5, 4.5))
plt.contourf(xx, yy, Z, levels=[-100, 0, 100], cmap="bwr", alpha=0.25)
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="bwr", s=18, edgecolors="k", linewidths=0.3)
plt.title("Decision boundary (MLP via Value)"); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 标量引擎的硬伤

训练区区 100 个点、几十步，就要花上一两秒——因为**每一个标量都是一个独立的图节点**：一次前向就要创建成千上万个 `Value` 对象，反向再逐个遍历。

真实网络动辄百万、上亿参数，标量引擎根本扛不住。解决办法是：**把一整个数组（张量）作为一个节点**，用 NumPy 的向量化一次算完一层——这就是下一站要造的 `Tensor` 引擎。

## 📦 沉淀进 minitorch

本节的 `Neuron / Layer / MLP` 是**基于标量 `Value` 的教学演示**，用来打通"搭网络 + 训练"的整体直觉，因此**不沉淀进包**。

真正可复用、面向张量的 `nn.Module / Linear`（API 贴近 PyTorch）将在 **Part 4** 基于 Part 3 的 `Tensor` 引擎构建。

## 小练习

1. **调网络结构**：把 `MLP(2, [16, 1])` 换成 `MLP(2, [16, 16, 1])`（更深），决策边界更平滑了吗？训练变慢了多少？
2. **学习率**：去掉学习率衰减（固定 `lr=1.0` 或 `lr=0.05`），观察收敛差异。
3. **感受"慢"**：把数据量 `n` 从 100 加到 500，记录训练时间的变化——它大致随样本数线性增长，这正是标量引擎的瓶颈。

## 小结 & 下一站

✅ 我们用 `Value` 搭出了 `Neuron / Layer / MLP`，用标准训练循环成功分类了非线性数据，并可视化了决策边界。

✅ 我们也亲身体会到：**标量自动求导虽然正确，但太慢**——因为它把每个数都当成图节点。

**下一站 → Part 3 `05_tensor_and_broadcasting`**：我们把 autograd 升级到**张量**版本。一个 `Tensor` 节点就包住一整个 NumPy 数组，用向量化一次算完一层。这是 minitorch 真正的"心脏"，后续 CNN / RNN / Transformer 全都建立在它之上。